In [1]:
import tensorflow_datasets as tfds
import tqdm
import numpy as np
from PIL import Image
from IPython import display
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.spatial.distance import euclidean
# from dtaidistance import dtw
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import pickle
import tensorflow_datasets as tfds
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import os




PATH = "/tsi/hi-paris/Pollen/datasets/tf_datasets/columbia_cairlab_pusht_real/0.1.0"

2026-02-13 15:40:24.856637: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-13 15:40:24.856710: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-13 15:40:24.858769: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-13 15:40:24.872273: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-13 15:40:26.245311: W tensorflow/comp

In [2]:
def disps_to_positions(disps):
    """
    disps: (T, 3) or (T, 2)
    returns positions: (T+1, D) with pos[0] = 0
    """
    # cumulative sum of displacements
    pos = np.concatenate(
        [np.zeros((1, disps.shape[1]), dtype=disps.dtype),
         np.cumsum(disps, axis=0)],
        axis=0
    )
    return pos

def rdp_segment_indices(points, eps):
    """
    points: (N, 2) or (N, 3)
    eps: max allowed perpendicular distance
    return: sorted list of indices to KEEP (key points), at least [0, N-1]
    """
    N = points.shape[0]
    
    def _rdp_rec(start, end, keep):
        if end <= start + 1:
            return
        
        p_start = points[start]
        p_end   = points[end]
        seg = p_end - p_start
        seg_norm = np.linalg.norm(seg)
        
        if seg_norm < 1e-9:
            # nearly identical points
            return
        
        # perpendicular distances to segment
        v = points[start+1:end] - p_start  # (M, D)
        # projection length (scalar) onto seg
        t = np.dot(v, seg) / (seg_norm**2)
        proj = np.outer(t, seg)            # (M, D)
        perp = v - proj
        dists = np.linalg.norm(perp, axis=1)
        
        max_idx = np.argmax(dists)
        max_dist = dists[max_idx]
        
        if max_dist > eps:
            # split at this point
            idx = start + 1 + max_idx
            keep.add(idx)
            _rdp_rec(start, idx, keep)
            _rdp_rec(idx, end, keep)
        else:
            # no need to split; straight enough
            return
    
    keep = {0, N-1}
    _rdp_rec(0, N-1, keep)
    return sorted(list(keep))

def segments_from_key_idxs(key_idxs, T):
    """
    key_idxs: indices over positions (0..T)
    T: number of displacements (positions length = T+1)
    returns: list of (start_t, end_t) for displacements
             where each segment is disps[start_t:end_t]  (end_t exclusive)
    """
    segments = []
    for a, b in zip(key_idxs[:-1], key_idxs[1:]):
        start_t = a
        end_t   = b  # because disps[t] moves from pos[t] -> pos[t+1]
        segments.append((start_t, end_t))
    return segments

def filter_segments(segments, disps,
                    min_len=5,
                    min_total_disp=0.01):
    """
    Remove tiny or too-short segments.
    disps: (T, 3)
    segments: list of (s,e)
    """
    good = []
    for (s, e) in segments:
        L = e - s
        if L < min_len:
            continue
        disp_vec = disps[s:e, :2].sum(axis=0)
        if np.linalg.norm(disp_vec) < min_total_disp:
            continue
        good.append((s, e))
    return good

def segment_direction_ok(disps, s, e, max_angle_deg=60):
    """
    Check that direction doesn't change too abruptly inside segment.
    """
    v = disps[s:e, :2]  # 2D
    norms = np.linalg.norm(v, axis=1, keepdims=True) + 1e-9
    u = v / norms
    dots = (u[:-1] * u[1:]).sum(axis=1)
    dots = np.clip(dots, -1.0, 1.0)
    angles = np.degrees(np.arccos(dots))
    return np.max(angles) <= max_angle_deg

def plot_trajectory_segments(disps, segments, title=None):
    """
    disps: (T, 3) array for a single episode
    segments: list of (start_t, end_t) indices
    """
    # Work in 2D for PushT
    pos = disps_to_positions(disps[:, :2])  # (T+1, 2)

    plt.figure(figsize=(6, 6))

    # Plot full trajectory (light)
    plt.plot(pos[:, 0], pos[:, 1], linestyle='--', alpha=0.3, label='full traj')

    # Color map for segments
    cmap = plt.get_cmap('tab20')
    num_seg = len(segments)

    for i, (s, e) in enumerate(segments):
        seg_pos = pos[s:e+1]  # positions along this segment
        color = cmap(i % 20)

        plt.plot(seg_pos[:, 0], seg_pos[:, 1], linewidth=2.5, color=color)

        # Mark start & end
        plt.scatter(seg_pos[0, 0], seg_pos[0, 1], marker='o', color=color)
        plt.scatter(seg_pos[-1, 0], seg_pos[-1, 1], marker='x', color=color)

        # Label segment id at its midpoint
        mid_idx = (s + e) // 2
        mid_pos = pos[mid_idx]
        plt.text(mid_pos[0], mid_pos[1], str(i),
                 fontsize=9, color=color,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.7))

    plt.gca().set_aspect('equal', adjustable='box')
    plt.xlabel('x (world)')
    plt.ylabel('y (world)')
    if title is not None:
        plt.title(title)
    else:
        plt.title('Trajectory with segmented subtrajectories')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

def visualize_episode_segments(episodes_displacements,
                               ep_idx,
                               eps=0.01,
                               filter=True,
                               enforce_smooth=True,
                               min_len=5,
                               min_total_disp=0.01,
                               max_angle_deg=60,method="rdp"):
    """
    episodes_displacements: dict[ep_idx] -> (T, 3)
    ep_idx: which episode to visualize
    """
    disps = episodes_displacements[ep_idx]  # (T, 3)
    T = disps.shape[0]

    # Positions for RDP (2D)
    positions = disps_to_positions(disps[:, :2])  # (T+1, 2)
    if method=="rdp":
        # 1) RDP to get keypoints
        key_idxs = rdp_segment_indices(positions, eps=eps)
        # 2) raw segments

        segments = segments_from_key_idxs(key_idxs, T)
        print("segments legth before filtering:", len(segments))

    elif method=="curvature":
            x = positions[:, 0]
            y = positions[:, 1]
            key_idxs=curvature_based_segmentation(x, y, vizualization=False)
            segments = segments_from_key_idxs(key_idxs, T)

    if filter:
    # 3) filter small/weak segments
        segments = filter_segments(segments, disps,
                                min_len=min_len,
                                min_total_disp=min_total_disp)
        print("segments length after filtering:", len(segments))

    if enforce_smooth:
        # 4) enforce direction smoothness
        final_segments = []
        for s, e in segments:
            if segment_direction_ok(disps, s, e, max_angle_deg=max_angle_deg):
                final_segments.append((s, e))
        segments = final_segments
        print("segments length after direction filtering:", len(segments))

    # 5) visualize
    title = f"Episode {ep_idx} — {len(segments)} segments"
    plot_trajectory_segments(disps, segments, title=title)

def curvature_based_segmentation(x, y, vizualization=False):
    """
    Segmente une trajectoire 2D définie par les coordonnées x et y en utilisant la courbure.
    
    Args:
        x (np.ndarray): Coordonnées x de la trajectoire.
        y (np.ndarray): Coordonnées y de la trajectoire.
        
    Returns:
        segment_indices (np.ndarray): Indices des points de coupure basés sur la courbure.
    """
    # --- 1. Préparation des Données ---    
    import numpy as np
    from scipy.signal import find_peaks
    import matplotlib.pyplot as plt


    path = np.array([x, y]).T
    N = len(path)

    # --- 2. Calcul de la Courbure (Approximation par Dérivées Numériques) ---

    # Calcul des dérivées premières (vitesse)
    dx = np.gradient(x)
    dy = np.gradient(y)

    # Calcul des dérivées secondes (accélération)
    ddx = np.gradient(dx)
    ddy = np.gradient(dy)

    # Formule de la Courbure Kappa (pour une courbe paramétrée en 2D)
    # kappa = |x'y'' - y'x''| / (x'^2 + y'^2)^(3/2)
    curvature = np.abs(dx * ddy - dy * ddx) / (dx**2 + dy**2)**(3/2)

    # Remplacer les valeurs NaN (aux extrémités où les dérivées sont moins précises)
    curvature = np.nan_to_num(curvature)

    # --- 3. Logique de Segmentation (Détection de Pics) ---

    # Trouver les indices où la courbure présente des pics significatifs.
    # On utilise un seuil (height) pour ne retenir que les virages nets.
    # La 'distance' entre les pics évite de détecter des pics voisins.
    peaks, _ = find_peaks(curvature, height=10, distance=5) 

    # Ajouter le point de départ et d'arrivée pour le découpage
    segment_indices = np.concatenate([[0], peaks, [N - 1]])
    segment_indices = np.unique(segment_indices) # Assure l'unicité et l'ordre



    # --- 4. Visualisation de la Segmentation ---
    if vizualization:
        # plt.figure(figsize=(10, 6))
        cmap = plt.get_cmap('tab20')
        plt.figure(figsize=(6, 6))
        plt.gca().set_aspect('equal', adjustable='box')
        plt.plot(x, y, color='lightblue', linestyle='--')
        plt.plot(x[segment_indices], y[segment_indices], 'ro')
        plt.plot(x, y, color='lightblue', linestyle='--')
        plt.plot(x[segment_indices], y[segment_indices], 'ro')

        # Tracer les segments (optionnel)
        for i in range(len(segment_indices) - 1):
            color = cmap(i % 20)
            start = segment_indices[i]
            end = segment_indices[i+1]
            plt.plot(x[start:end+1], y[start:end+1], linewidth=3, alpha=0.7,color=color)
            mid_idx = (start + end) // 2
            mid_posx = x[mid_idx]
            mid_posy = y[mid_idx]

            plt.text(mid_posx, mid_posy, str(i),
                        fontsize=9, color=color,
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=color, alpha=0.7))

        plt.title("Segmentation Basée sur la Courbure")
        plt.xlabel("Position X")
        plt.ylabel("Position Y")
        plt.legend()
        plt.grid(True)
        plt.show()
        

        print(f"Indices de coupure détectés : {segment_indices}")
    
    return segment_indices

# Extract all subtrajectories from the dataset

def extract_all_subtrajectories(episodes_displacements, eps=0.01, filter=False,
                               enforce_smooth=False,
                               min_len=5,
                               min_total_disp=0.01,
                               max_angle_deg=60, use_xy_only=True,method="DP"):
    """
    episodes_displacements: dict[ep_idx] -> (T, 3) displacements
    eps: RDP epsilon
    use_xy_only: if True, segment in 2D (x,y) space

    returns:
      subtrajs: list of np.ndarrays of shape (L_i, D)  (displacements)
      meta:     list of dicts with episode index and time indices for each subtraj
    """
    all_subtrajs = []
    all_meta     = []

    for ep_idx, disps in episodes_displacements.items():
        T, D = disps.shape

        # Positions for RDP (2D or 3D)
        if use_xy_only:
            positions = disps_to_positions(disps[:, :2])  # (T+1, 2)
        else:
            positions = disps_to_positions(disps)          # (T+1, D)

        if method=="DP":
        # 1) RDP keypoints
            key_idxs = rdp_segment_indices(positions, eps=eps)

            # 2) time segments
            segments = segments_from_key_idxs(key_idxs, T)

            if filter:
            # 3) filter small/weak segments
                segments = filter_segments(segments, disps,
                                        min_len=min_len,
                                        min_total_disp=min_total_disp)
                #print("segments length after filtering:", len(segments))

            if enforce_smooth:
                # 4) enforce direction smoothness
                final_segments = []
                for s, e in segments:
                    if segment_direction_ok(disps, s, e, max_angle_deg=max_angle_deg):
                        final_segments.append((s, e))
                segments = final_segments
                print("segments length after direction filtering:", len(segments))


        elif method=="curvature":
            x = positions[:, 0]
            y = positions[:, 1]
            key_idxs=curvature_based_segmentation(x, y, vizualization=False)
            segments = segments_from_key_idxs(key_idxs, T)
  


        # 3) collect subtrajectories
        for (s, e) in segments:
            subtraj = disps[s:e]   # (L, D)
            all_subtrajs.append(subtraj)
            all_meta.append({
                "episode": ep_idx,
                "start_t": s,
                "end_t": e
            })

    return all_subtrajs, all_meta

# --- Feature extraction helpers ---
def center_and_align(subtraj):
    """
    Center subtrajectory at origin and rotate so start->end aligns with +X axis.
    subtraj: (L, D) array
    returns: (L, D) aligned array
    """
    # Center
    centered = subtraj - subtraj[0]
    # Compute start->end vector
    v = centered[-1] - centered[0]
    norm = np.linalg.norm(v)
    if norm < 1e-9:
        return centered  # degenerate, no rotation
    # Only rotate in XY
    angle = np.arctan2(v[1], v[0])
    rot = np.array([[np.cos(-angle), -np.sin(-angle)], [np.sin(-angle), np.cos(-angle)]])
    if centered.shape[1] == 2:
        aligned = centered @ rot.T
    else:
        rot3d = np.eye(3)
        rot3d[:2, :2] = rot
        aligned = centered @ rot3d.T
    return aligned

def compute_features(subtraj):
    """
    Compute features: linearity, length, orientation, barycenter.
    subtraj: (L, D) array
    returns: feature vector (shape, amplitude, direction, barycenter)
    """
    aligned = center_and_align(subtraj)
    # Positions
    pos = np.vstack([np.zeros(aligned.shape[1]), np.cumsum(aligned, axis=0)])
    start = pos[0]
    end = pos[-1]
    direct_dist = np.linalg.norm(end - start)
    arc_length = np.sum(np.linalg.norm(np.diff(pos, axis=0), axis=1))
    linearity = direct_dist / arc_length if arc_length > 1e-9 else 0.0
    length = arc_length
    orientation = (end - start) / (np.linalg.norm(end - start) + 1e-9)
    barycenter = pos.mean(axis=0)
    # Compose feature vector
    features = np.concatenate([
        [linearity],
        [length],
        orientation,
        barycenter
    ])
    return features

def normalize_subtraj(subtraj):
    """
    Normalize a subtrajectory to unit length and zero-mean.
    subtraj: (L, D) array
    returns: (L, D) normalized array
    """
    # Zero-mean per dimension
    subtraj = subtraj - subtraj.mean(axis=0)
    # Unit norm (L2 norm over all elements)
    norm = np.linalg.norm(subtraj)
    if norm > 1e-9:
        subtraj = subtraj / norm
    return subtraj

def pad_subtraj(subtraj, max_len):
    """
    Pad a subtrajectory to max_len by repeating last row.
    subtraj: (L, D) array
    max_len: target length
    returns: (max_len, D) padded array
    """
    if subtraj.shape[0] >= max_len:
        return subtraj[:max_len]
    pad_rows = max_len - subtraj.shape[0]
    padding = np.repeat(subtraj[-1:], pad_rows, axis=0)
    return np.vstack([subtraj, padding])

def compute_pairwise_dtw_distance(subtrajs, normalized=True, max_len=None):
    """
    Compute pairwise DTW distance between subtrajectories.
    subtrajs: list of (L_i, D) arrays
    normalized: if True, normalize each subtrajectory first
    max_len: if provided, pad all to this length; else use max length in dataset
    returns: (N, N) distance matrix (symmetric)
    """
    N = len(subtrajs)
    
    if max_len is None:
        max_len = max([s.shape[0] for s in subtrajs])
    
    # Preprocess: normalize and pad
    processed = []
    for s in subtrajs:
        if normalized:
            s = normalize_subtraj(s)
        s = pad_subtraj(s, max_len)
        processed.append(s.flatten())  # flatten to 1D for DTW
    
    # Compute pairwise DTW distances
    dist_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            d = dtw.distance(processed[i], processed[j])
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d
    
    return dist_matrix

def compute_pairwise_euclidean_distance(subtrajs, normalized=True, max_len=None):
    """
    Compute pairwise Euclidean distance between subtrajectories.
    Simpler and faster than DTW; works well for similar-length trajectories.
    subtrajs: list of (L_i, D) arrays
    normalized: if True, normalize each subtrajectory first
    max_len: if provided, pad all to this length
    returns: (N, N) distance matrix (symmetric)
    """
    N = len(subtrajs)
    
    if max_len is None:
        max_len = max([s.shape[0] for s in subtrajs])
    
    # Preprocess: normalize, pad, flatten
    processed = []
    for s in subtrajs:
        if normalized:
            s = normalize_subtraj(s)
        s = pad_subtraj(s, max_len)
        processed.append(s.flatten())
    
    processed = np.array(processed)  # (N, max_len*D)
    
    # Compute pairwise Euclidean distances
    from scipy.spatial.distance import pdist, squareform
    dist_matrix = squareform(pdist(processed, metric='euclidean'))
    
    return dist_matrix

def cluster_subtrajectories(all_subtrajs, all_meta, n_clusters=10, method='euclidean', normalized=True, max_len=None):
    """
    Cluster subtrajectories using KMeans on pairwise distances.
    all_subtrajs: list of (L_i, D) arrays (subtrajectories)
    all_meta: list of dicts with metadata
    n_clusters: number of clusters
    method: 'euclidean' or 'dtw'
    normalized: if True, normalize before distance computation
    max_len: if provided, pad all to this length
    
    returns:
      cluster_labels: (N,) array of cluster assignments
      cluster_centers_idx: list of indices of closest subtraj to each cluster center
      dist_matrix: (N, N) pairwise distance matrix
    """
    print(f"Computing {method} distances for {len(all_subtrajs)} subtrajectories...")
    
    if method == 'euclidean':
        dist_matrix = compute_pairwise_euclidean_distance(all_subtrajs, normalized=normalized, max_len=max_len)
    elif method == 'dtw':
        dist_matrix = compute_pairwise_dtw_distance(all_subtrajs, normalized=normalized, max_len=max_len)
    elif method=='featured':
        features = np.array([compute_features(s) for s in all_subtrajs])

        # Scale features if desired
        if normalized:   # or scale_features
            scaler = StandardScaler()
            features = scaler.fit_transform(features)

        # Keep name 'dist_matrix' for compatibility but here it is not a distance matrix
        dist_matrix = features
    else:
        raise ValueError(f"Unknown method: {method}")
    
    print(f"Clustering {len(all_subtrajs)} subtrajectories into {n_clusters} clusters...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(dist_matrix)
    
    # Find representative (closest to center) for each cluster
    cluster_centers_idx = []
    for cluster_id in range(n_clusters):
        member_indices = np.where(cluster_labels == cluster_id)[0]
        if len(member_indices) > 0:
            # Find member closest to cluster center
            if method == 'featured':
                distances = kmeans.transform(features)
                closest_idx =member_indices[np.argmin(distances[member_indices, cluster_id])]
                cluster_centers_idx.append(closest_idx)
            else:
                center = dist_matrix[member_indices][:, member_indices].mean(axis=1)
                closest_idx = member_indices[np.argmin(center)]
                cluster_centers_idx.append(closest_idx)
        
    return cluster_labels, cluster_centers_idx, dist_matrix

def cluster_subtrajectories_fit(all_subtrajs, all_meta, n_clusters=10, method='euclidean', normalized=True, max_len=None):
    """
    Cluster subtrajectories using KMeans on pairwise distances.
    all_subtrajs: list of (L_i, D) arrays (subtrajectories)
    all_meta: list of dicts with metadata
    n_clusters: number of clusters
    method: 'euclidean' or 'dtw'
    normalized: if True, normalize before distance computation
    max_len: if provided, pad all to this length
    
    returns:
      cluster_labels: (N,) array of cluster assignments
      cluster_centers_idx: list of indices of closest subtraj to each cluster center
      dist_matrix: (N, N) pairwise distance matrix
    """
    print(f"Computing {method} distances for {len(all_subtrajs)} subtrajectories...")
    
    if method == 'euclidean':
        dist_matrix = compute_pairwise_euclidean_distance(all_subtrajs, normalized=normalized, max_len=max_len)
    elif method == 'dtw':
        dist_matrix = compute_pairwise_dtw_distance(all_subtrajs, normalized=normalized, max_len=max_len)
    elif method=='featured':
        features = np.array([compute_features(s) for s in all_subtrajs])

        # Scale features if desired
        if normalized:   # or scale_features
            scaler = StandardScaler()
            features = scaler.fit_transform(features)

        # Keep name 'dist_matrix' for compatibility but here it is not a distance matrix
        dist_matrix = features
    else:
        raise ValueError(f"Unknown method: {method}")
    
    print(f"Clustering {len(all_subtrajs)} subtrajectories into {n_clusters} clusters...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    return kmeans.fit(dist_matrix)
    
   

def visualize_clusters(all_subtrajs, cluster_labels, n_samples=None,method="t-SNE",normalized=True):
    """
    Visualize clusters using t-SNE on flattened, normalized subtrajectories.
    all_subtrajs: list of (L_i, D) arrays
    cluster_labels: (N,) cluster assignments
    n_samples: if provided, subsample this many subtrajectories for visualization
    """
    N = len(all_subtrajs)
    
    if n_samples is not None and n_samples < N:
        indices = np.random.choice(N, n_samples, replace=False)
    else:
        indices = np.arange(N)
    
    max_len = max([all_subtrajs[i].shape[0] for i in indices])
    
    # Prepare data for t-SNE
    processed = []
    labels_sub = []
    if normalized:
        for i in indices:
            s = normalize_subtraj(all_subtrajs[i])
            s = pad_subtraj(s, max_len)
            processed.append(s.flatten())
            labels_sub.append(cluster_labels[i])
        
        processed = np.array(processed)
        labels_sub = np.array(labels_sub)
    else:
        processed=all_subtrajs
        labels_sub=cluster_labels
    
    if method=="t-SNE":
        # t-SNE embedding
        print("Running t-SNE for visualization...")
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(processed) - 1))
        embedding = tsne.fit_transform(processed)
        
        # Plot
        plt.figure(figsize=(10, 8))
        scatter = plt.scatter(embedding[:, 0], embedding[:, 1], c=labels_sub, cmap='tab20', s=100, alpha=0.7)
        plt.colorbar(scatter, label='Cluster ID')
        plt.xlabel('t-SNE 1')
        plt.ylabel('t-SNE 2')
        plt.title(f'Clustered Subtrajectories (n={len(indices)})')
        plt.grid(True)
        plt.tight_layout()
        plt.show()
    elif method=="PCA":
        from sklearn.decomposition import PCA
        pca = PCA(n_components=2)
        embedding = pca.fit_transform(processed)
        
        # Plot
        plt.figure(figsize=(10, 8))
        scatter = plt.scatter(embedding[:, 0], embedding[:, 1], c=labels_sub, cmap='tab20', s=100, alpha=0.7)
        plt.colorbar(scatter, label='Cluster ID')
        plt.xlabel('PCA 1')
        plt.ylabel('PCA 2')
        plt.title(f'Clustered Subtrajectories (n={len(indices)})')
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        

def print_cluster_summary(all_subtrajs, all_meta, cluster_labels, cluster_centers_idx):
    """
    Print summary of each cluster: size, representatives, episodes.
    """
    n_clusters = len(cluster_centers_idx)
    
    print(f"\n{'='*80}")
    print(f"Cluster Summary ({n_clusters} clusters)")
    print(f"{'='*80}\n")
    
    for cluster_id in range(n_clusters):
        member_indices = np.where(cluster_labels == cluster_id)[0]
        print(f"Cluster {cluster_id}: {len(member_indices)} members")
        
        # Show representative
        rep_idx = cluster_centers_idx[cluster_id]
        rep_meta = all_meta[rep_idx]
        rep_shape = all_subtrajs[rep_idx].shape
        print(f"  Representative: subtrajectory {rep_idx}, episode {rep_meta['episode']}, shape {rep_shape}")
        
        # Show episode distribution
        ep_counts = {}
        for idx in member_indices:
            ep = all_meta[idx]['episode']
            ep_counts[ep] = ep_counts.get(ep, 0) + 1
        print(f"  Episode distribution: {dict(sorted(ep_counts.items())[:5])}...")  # show first 5 episodes
        print()




In [3]:
import json
from tensorflow.keras.preprocessing.sequence import pad_sequences

features = tfds.features.FeaturesDict({
            "steps": tfds.features.Sequence(tfds.features.FeaturesDict({
                "action": tfds.features.FeaturesDict({
                    "gripper_closedness_action": tfds.features.Scalar(dtype=tf.float32),
                    "rotation_delta": tfds.features.Tensor(shape=(3,), dtype=tf.float32),
                    "terminate_episode": tfds.features.Scalar(dtype=tf.float32),
                    "world_vector": tfds.features.Tensor(shape=(3,), dtype=tf.float32),
                }),
                "is_first": tfds.features.Scalar(dtype=tf.bool),
                "is_last": tfds.features.Scalar(dtype=tf.bool),
                "is_terminal": tfds.features.Scalar(dtype=tf.bool),
                "observation": tfds.features.FeaturesDict({
                    "image": tfds.features.Image(shape=(240, 320, 3), dtype=tf.uint8),
                    "natural_language_embedding": tfds.features.Tensor(shape=(512,), dtype=tf.float32),
                    "natural_language_instruction": tfds.features.Text(),
                    "robot_state": tfds.features.Tensor(shape=(2,), dtype=tf.float32),
                    "wrist_image": tfds.features.Image(shape=(240, 320, 3), dtype=tf.uint8),
                    "cluster_id": tfds.features.Scalar(dtype=tf.int64),

                }),
                "reward": tfds.features.Scalar(dtype=tf.float32),
            })),
        })




class TrajectoryTokenizer:
    def __init__(self, n_clusters=64, random_state=42,method="featured",normalized=True):
        self.n_clusters = n_clusters
        self.kmeans = None
        self.cluster_centers_ = None
        self.n_features_ = None
        self.method=method
        self.normalized=normalized

    def fit(self, list_of_trajectories):
        self.kmeans = cluster_subtrajectories_fit(
            list_of_trajectories, 
            all_meta, 
            n_clusters=self.n_clusters,
            method=self.method,  # 'euclidean' is faster than 'dtw'
            normalized=True,
            max_len=None  # will auto-pad to longest subtraj
        )
        # self.cluster_centers_ = self.kmeans.cluster_centers_
        # self.n_features_ = dist_matrix.shape[1]
        return self

    def encode(self, pts):
        """Return token id (cluster) for a single segment (T,3)"""
        if self.method=="featured":
            fz = compute_features([pts])
        else:
            fz=[pts]
        return int(self.kmeans.predict(fz)[0])

    def batch_encode(self, list_of_segments):
        if self.method=="featured":
            Fz = np.array([compute_features(s) for s in all_subtrajs])
                  # Scale features if desired
            if self.normalized:   # or scale_features
                scaler = StandardScaler()
                Fz = scaler.fit_transform(Fz)
        else:
            Fz=list_of_segments
        # print(Fz)
        # Fz = pad_sequences(Fz, padding='post', dtype='float32')
        # print(Fz)
        ids = self.kmeans.predict(Fz)
        return ids

    def save(self, path_json):
        meta = {
            "n_clusters": int(self.n_clusters),
            # "cluster_centers": self.cluster_centers_.tolist(),
            # "n_features": int(self.n_features_)
        }
        with open(path_json, "w") as f:
            json.dump(meta, f, indent=2)

    def load(self, path_json):
        with open(path_json, "r") as f:
            meta = json.load(f)
        self.n_clusters = meta["n_clusters"]
        # self.cluster_centers_ = np.array(meta["cluster_centers"])
        # self.n_features_ = meta["n_features"]
        # reconstruct kmeans-like object (predict only using centers)
        class DummyKMeans:
            def __init__(self, centers):
                self.cluster_centers_ = centers
            def predict(self, X):
                # assign to nearest center (euclidean)
                d = np.linalg.norm(X[:, None, :] - self.cluster_centers_[None, :, :], axis=2)
                return np.argmin(d, axis=1)
        self.kmeans = DummyKMeans(self.cluster_centers_)
        return self


class MyDataset(tfds.core.GeneratorBasedBuilder):
    VERSION = tfds.core.Version('1.0.0')
    # On force manuellement le nom pour éviter que TFDS le cherche via le fichier source
    name = "columbia_cairlab_pusht_real"

    def _info(self):
        return tfds.core.DatasetInfo(
            builder=self,
            features=features, # Ton FeaturesDict
            description="Fix interactif",
        )

    def _split_generators(self, dl_manager):
        # On définit le split sans passer par dl_manager qui cause le crash
        return {'train': self._generate_examples()}

    def _generate_examples(self):
        # On utilise directement ds et mappingbis qui sont dans le scope global
        print("\n[DEBUG] Le générateur a démarré !")
        for ep_idx, episode in enumerate(tqdm(ds)):
            print(f"\n[DEBUG] Traitement de l'épisode {ep_idx}...")
            ep_data = tfds.as_numpy(episode)
            steps_list = list(ep_data['steps'])
            cluster_ids = mappingbis[ep_idx]

            formatted_steps = {
                "is_first": np.array([s["is_first"] for s in steps_list], dtype=np.bool_),
                "is_last": np.array([s["is_last"] for s in steps_list], dtype=np.bool_),
                "is_terminal": np.array([s["is_terminal"] for s in steps_list], dtype=np.bool_),
                "reward": np.array([s["reward"] for s in steps_list], dtype=np.float32),
                "action": {
                    "gripper_closedness_action": np.array([s["action"]["gripper_closedness_action"] for s in steps_list], dtype=np.float32),
                    "rotation_delta": np.array([s["action"]["rotation_delta"] for s in steps_list], dtype=np.float32),
                    "terminate_episode": np.array([s["action"]["terminate_episode"] for s in steps_list], dtype=np.float32),
                    "world_vector": np.array([s["action"]["world_vector"] for s in steps_list], dtype=np.float32),
                },
                "observation": {
                    "image": np.array([s["observation"]["image"] for s in steps_list], dtype=np.uint8),
                    "natural_language_embedding": np.array([s["observation"]["natural_language_embedding"] for s in steps_list], dtype=np.float32),
                    "natural_language_instruction": [
                        s["observation"]["natural_language_instruction"].decode("utf-8") 
                        if isinstance(s["observation"]["natural_language_instruction"], bytes) 
                        else str(s["observation"]["natural_language_instruction"]) 
                        for s in steps_list
                    ],
                    "robot_state": np.array([s["observation"]["robot_state"] for s in steps_list], dtype=np.float32),
                    "wrist_image": np.array([s["observation"]["wrist_image"] for s in steps_list], dtype=np.uint8),
                    "cluster_id": np.array(cluster_ids, dtype=np.int64),

                   
                }
            }
            yield ep_idx, {"steps": formatted_steps}
            print(f"[DEBUG] Épisode {ep_idx} envoyé au writer.")




# SUBTRAJECTORY AND CLUSTERING

In [4]:
# Load builder from local directory
builder = tfds.builder_from_directory(builder_dir=PATH)

# Full train split; you can slice if you want (e.g. "train[:10]")
ds = builder.as_dataset(split="train")

episodes_displacements = {}  # ep_idx -> dict with 'instruction' and 'displacements'

mapping=[]


for ep_idx, episode in enumerate(ds):
    # episode["steps"] is a tf.data.Dataset of step dicts
    step_ds = episode["steps"]  
    disps = []
    temp=[]
    mapping.append(temp)
    st_idx= 0

    # Convert steps to numpy for easy use
    for  step in tfds.as_numpy(step_ds):
      # world_vector is shape (3,) float32: [dx, dy, dz]
        dv = step["action"]["world_vector"]  # numpy array of shape (3,)

        disps.append(dv)
        mapping[ep_idx].append(dv)
        st_idx += 1

    # Convert list of (3,) arrays to (T, 3) array
    disps = np.stack(disps, axis=0)  # shape (T, 3)

    episodes_displacements[ep_idx] = disps



all_subtrajs, all_meta = extract_all_subtrajectories(episodes_displacements, eps=0.01, use_xy_only=True, method="curvature")
print(len(all_subtrajs), "subtrajectories total")
print(all_meta[0], all_subtrajs[0].shape)
print(all_subtrajs[0])
print(mapping[0])


2026-02-13 15:40:42.142447: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1045 MB memory:  -> device: 0, name: NVIDIA A40, pci bus id: 0000:d8:00.0, compute capability: 8.6


2662 subtrajectories total
{'episode': 0, 'start_t': 0, 'end_t': 14} (14, 3)
[[-2.35729181e-04  3.46552079e-05  0.00000000e+00]
 [-6.43605745e-05 -6.13791822e-03  0.00000000e+00]
 [-3.95408459e-03 -2.00672355e-02  0.00000000e+00]
 [-6.87524024e-03 -2.51260176e-02  0.00000000e+00]
 [-8.40447098e-03 -2.76479200e-02  0.00000000e+00]
 [-1.11666108e-02 -2.74375230e-02  0.00000000e+00]
 [-1.22035788e-02 -2.83297393e-02  0.00000000e+00]
 [-1.32332025e-02 -2.94659697e-02  0.00000000e+00]
 [-1.51430368e-02 -2.70028654e-02  0.00000000e+00]
 [-1.60367917e-02 -2.80755125e-02  0.00000000e+00]
 [-1.46422312e-02 -2.75504328e-02  0.00000000e+00]
 [-1.76687930e-02 -2.83966921e-02  0.00000000e+00]
 [-1.57611817e-02 -2.50646230e-02  0.00000000e+00]
 [-7.65535887e-03 -2.46636420e-02  0.00000000e+00]]
[array([-2.3572918e-04,  3.4655208e-05,  0.0000000e+00], dtype=float32), array([-6.4360574e-05, -6.1379182e-03,  0.0000000e+00], dtype=float32), array([-0.00395408, -0.02006724,  0.        ], dtype=float32), 

In [5]:
# tokkenize all segments
tok = TrajectoryTokenizer(n_clusters=30, method="featured", normalized=True)
tok.fit(all_subtrajs)
# encode all segments and show distribution
ids = tok.batch_encode(all_subtrajs)
uniq, counts = np.unique(ids, return_counts=True)
print("Cluster counts:", dict(zip(uniq.tolist(), counts.tolist())))
# save
tok.save("trajectory_vocab.json")
print("Saved vocab -> trajectory_vocab.json")

# inspect cluster counts
for i in range(30):
    count = np.sum(ids == i)
    print(f"Cluster {i}: {count} trajectories")



Computing featured distances for 2662 subtrajectories...
Clustering 2662 subtrajectories into 30 clusters...
Cluster counts: {0: 138, 1: 93, 2: 106, 3: 40, 4: 41, 5: 243, 6: 8, 7: 557, 8: 21, 9: 92, 10: 14, 11: 59, 12: 45, 13: 6, 14: 76, 15: 139, 16: 25, 17: 30, 18: 17, 19: 342, 20: 53, 21: 1, 22: 19, 23: 11, 24: 94, 25: 191, 26: 79, 27: 9, 28: 43, 29: 70}
Saved vocab -> trajectory_vocab.json
Cluster 0: 138 trajectories
Cluster 1: 93 trajectories
Cluster 2: 106 trajectories
Cluster 3: 40 trajectories
Cluster 4: 41 trajectories
Cluster 5: 243 trajectories
Cluster 6: 8 trajectories
Cluster 7: 557 trajectories
Cluster 8: 21 trajectories
Cluster 9: 92 trajectories
Cluster 10: 14 trajectories
Cluster 11: 59 trajectories
Cluster 12: 45 trajectories
Cluster 13: 6 trajectories
Cluster 14: 76 trajectories
Cluster 15: 139 trajectories
Cluster 16: 25 trajectories
Cluster 17: 30 trajectories
Cluster 18: 17 trajectories
Cluster 19: 342 trajectories
Cluster 20: 53 trajectories
Cluster 21: 1 trajecto

In [6]:
# After computing cluster IDs with tok.batch_encode()

# 1. Initialize mappingbis with zeros for each episode
mappingbis = []
for ep_idx, episode in enumerate(ds):
    step_ds = episode["steps"]
    num_steps = len(list(tfds.as_numpy(step_ds)))
    mappingbis.append([-1] * num_steps)  # One entry per action

# 2. Map subtrajectory cluster IDs back to action indices
print("Mapping subtrajectory cluster IDs to action indices...")
for subtraj_idx, meta in enumerate(all_meta):
    cluster_id = ids[subtraj_idx]  # The cluster ID for this subtrajectory
    ep_idx = meta["episode"]
    start_t = meta["start_t"]
    end_t = meta["end_t"]
    
    # Assign this cluster ID to all actions in this subtrajectory
    for action_idx in range(start_t, end_t):
        mappingbis[ep_idx][action_idx] = cluster_id

print("✅ mappingbis populated with cluster IDs")

# 3. Verify coverage
print("\nVerifying coverage...")
for ep_idx, episode_clusters in enumerate(mappingbis):
    covered = sum(1 for c in episode_clusters if c !=-1)
    total = len(episode_clusters)
    print(f"Episode {ep_idx}: {covered}/{total} actions covered ({100*covered/total:.1f}%)")

# 4. Save
import pickle
with open("cluster_ids.pkl", "wb") as f:
    pickle.dump(mappingbis, f)
print("\n✅ Saved cluster_ids.pkl")

Mapping subtrajectory cluster IDs to action indices...
✅ mappingbis populated with cluster IDs

Verifying coverage...
Episode 0: 163/163 actions covered (100.0%)
Episode 1: 188/188 actions covered (100.0%)
Episode 2: 271/271 actions covered (100.0%)
Episode 3: 158/158 actions covered (100.0%)
Episode 4: 200/200 actions covered (100.0%)
Episode 5: 237/237 actions covered (100.0%)
Episode 6: 174/174 actions covered (100.0%)
Episode 7: 177/177 actions covered (100.0%)
Episode 8: 236/236 actions covered (100.0%)
Episode 9: 125/125 actions covered (100.0%)
Episode 10: 227/227 actions covered (100.0%)
Episode 11: 195/195 actions covered (100.0%)
Episode 12: 216/216 actions covered (100.0%)
Episode 13: 210/210 actions covered (100.0%)
Episode 14: 232/232 actions covered (100.0%)
Episode 15: 262/262 actions covered (100.0%)
Episode 16: 127/127 actions covered (100.0%)
Episode 17: 205/205 actions covered (100.0%)
Episode 18: 217/217 actions covered (100.0%)
Episode 19: 279/279 actions covered (

In [7]:
output_dir = "/home/ids/ext-5219/tokenizer/test"

builder = MyDataset(data_dir=output_dir)

# On écrase l'attribut qui pose problème par None (pas une property, juste None)
builder.__class__.code_path = None

# Charger
with open("cluster_ids.pkl", "rb") as f:
    mappingbis = pickle.load(f)


# On appelle la méthode de bas niveau qui génère les TFRecords 
# en sautant la phase de "Download" qui cherche les checksums.
builder._download_and_prepare(
    dl_manager=tfds.download.DownloadManager(download_dir=output_dir, force_download=False),
    download_config=tfds.download.DownloadConfig(register_checksums=False)
)

print(f"Dataset terminé dans {builder.data_dir}")

# Chemin exact où se trouvent tes tfrecords (le dossier 1.0.0)
final_path = os.path.join(output_dir, "columbia_cairlab_pusht_real", "1.0.0")

# On force l'écriture du fichier metadata
builder.info.write_to_directory(final_path)

print(f"Fichier dataset_info.json généré dans {final_path}")

/home/ids/ext-5219/miniconda3/envs/openvla-oft/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating splits...:   0%| | 0


[DEBUG] Le générateur a démarré !



[DEBUG] Traitement de l'épisode 0...


[DEBUG] Épisode 0 envoyé au writer.

[DEBUG] Traitement de l'épisode 1...


[DEBUG] Épisode 1 envoyé au writer.

[DEBUG] Traitement de l'épisode 2...


...

[DEBUG] Épisode 119 envoyé au writer.

[DEBUG] Traitement de l'épisode 120...


[DEBUG] Épisode 120 envoyé au writer.

[DEBUG] Traitement de l'épisode 121...



100%|█| 122/122 [25:22<00:00, 1


[DEBUG] Épisode 121 envoyé au writer.


Dataset terminé dans /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0
Fichier dataset_info.json généré dans /home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0


# DIAGNOSTIC

In [14]:
import tensorflow_datasets as tfds
import os

data_dir = "/home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0/"

# 1. Charger le builder depuis le dossier
builder = tfds.builder_from_directory(data_dir)

# 2. Inspecter les features (pour vérifier la structure)
print("Structure des features :")
print(builder.info.features)

# 3. Charger les données et vérifier un échantillon
ds_test = builder.as_dataset(split='train')

for episode in ds_test.take(1):
    steps = episode['steps']
    print("\n--- Vérification du premier épisode ---")
    print(f"Nombre de steps : {len(steps['reward'])}")
    print(f"Clés disponibles dans l'observation : {steps['observation'].keys()}")
    
    # VERIFICATION CRUCIALE : Tes cluster_ids
    cluster_ids = steps['observation']['cluster_id'].numpy()
    print(f"Cluster IDs (5 premiers) : {cluster_ids[:5]}")
    
    # Vérification des images
    print(f"Format de l'image : {steps['observation']['image'].shape}")

Structure des features :
FeaturesDict({
    'steps': Sequence({
        'action': FeaturesDict({
            'gripper_closedness_action': Scalar(shape=(), dtype=float32),
            'rotation_delta': Tensor(shape=(3,), dtype=float32),
            'terminate_episode': Scalar(shape=(), dtype=float32),
            'world_vector': Tensor(shape=(3,), dtype=float32),
        }),
        'is_first': Scalar(shape=(), dtype=bool),
        'is_last': Scalar(shape=(), dtype=bool),
        'is_terminal': Scalar(shape=(), dtype=bool),
        'observation': FeaturesDict({
            'cluster_id': Scalar(shape=(), dtype=int64),
            'image': Image(shape=(240, 320, 3), dtype=uint8),
            'natural_language_embedding': Tensor(shape=(512,), dtype=float32),
            'natural_language_instruction': Text(shape=(), dtype=string),
            'robot_state': Tensor(shape=(2,), dtype=float32),
            'wrist_image': Image(shape=(240, 320, 3), dtype=uint8),
        }),
        'reward': S

In [18]:
# Trouver les moments de transition entre les clusters
diffs = np.where(np.diff(cluster_ids) != 0)[0]
if len(diffs) > 0:
    print(f"Changements de cluster détectés aux steps : {diffs}")
    for idx in diffs:
        print(f"Step {idx}: Cluster {cluster_ids[idx]} -> Step {idx+1}: Cluster {cluster_ids[idx+1]}")
else:
    print("Toutes les actions de cet épisode appartiennent au même cluster.")

Changements de cluster détectés aux steps : [ 12  19  38  46  51  65  72  78  86  91  97 102 108 116 125 136 141 149
 159 165 170 175 180 187 200 232 238 244 250 260 269]
Step 12: Cluster 26 -> Step 13: Cluster 5
Step 19: Cluster 5 -> Step 20: Cluster 24
Step 38: Cluster 24 -> Step 39: Cluster 5
Step 46: Cluster 5 -> Step 47: Cluster 19
Step 51: Cluster 19 -> Step 52: Cluster 0
Step 65: Cluster 0 -> Step 66: Cluster 7
Step 72: Cluster 7 -> Step 73: Cluster 5
Step 78: Cluster 5 -> Step 79: Cluster 2
Step 86: Cluster 2 -> Step 87: Cluster 5
Step 91: Cluster 5 -> Step 92: Cluster 19
Step 97: Cluster 19 -> Step 98: Cluster 25
Step 102: Cluster 25 -> Step 103: Cluster 7
Step 108: Cluster 7 -> Step 109: Cluster 24
Step 116: Cluster 24 -> Step 117: Cluster 25
Step 125: Cluster 25 -> Step 126: Cluster 3
Step 136: Cluster 3 -> Step 137: Cluster 2
Step 141: Cluster 2 -> Step 142: Cluster 24
Step 149: Cluster 24 -> Step 150: Cluster 25
Step 159: Cluster 25 -> Step 160: Cluster 5
Step 165: Cluster

In [2]:
import tensorflow_datasets as tfds
import numpy as np

data_dir = "/home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0/"

# Charger et vérifier
builder = tfds.builder_from_directory(data_dir)
ds_test = builder.as_dataset(split='train')

for ep_idx, episode in enumerate(ds_test.take(1)):
    steps = episode['steps']
    cluster_ids = steps['observation']['cluster_id'].numpy()
    
    print(f"Episode {ep_idx}:")
    print(f"  Cluster IDs shape: {cluster_ids.shape}")
    print(f"  Unique cluster IDs: {np.unique(cluster_ids)}")
    print(f"  Min: {np.min(cluster_ids)}, Max: {np.max(cluster_ids)}")
    print(f"  Data type: {cluster_ids.dtype}")
    
    # Vérifier qu'il n'y a pas de valeurs aberrantes
    if np.max(cluster_ids) >= 30:
        print(f"  ⚠️  ERROR: Cluster ID {np.max(cluster_ids)} >= 30!")

Episode 0:
  Cluster IDs shape: (221,)
  Unique cluster IDs: [ 0  1  2  3  7 10 12 14 15 19 25 26 29]
  Min: 0, Max: 29
  Data type: int64


In [ ]:
import tensorflow_datasets as tfds
import numpy as np
import pickle

print("="*80)
print("DIAGNOSTIC COMPLET DU DATASET")
print("="*80)

# Charger mappingbis depuis le pickle
try:
    with open("cluster_ids.pkl", "rb") as f:
        mappingbis = pickle.load(f)
    print("✅ mappingbis chargé depuis cluster_ids.pkl")
except:
    print("⚠️  Impossible de charger cluster_ids.pkl - réexecutez les cellules précédentes d'abord!")
    raise

# Recharger le dataset original
PATH = "/tsi/hi-paris/Pollen/datasets/tf_datasets/columbia_cairlab_pusht_real/0.1.0"
builder_original = tfds.builder_from_directory(builder_dir=PATH)
ds = builder_original.as_dataset(split="train")

# 1. Vérifier la couverture des segments
print("\n[1] VÉRIFICATION DE LA COUVERTURE DES SEGMENTS")
print("-" * 80)

total_actions = 0
total_covered_actions = 0
episode_details = []

for ep_idx, episode in enumerate(ds):
    step_ds = episode["steps"]
    steps_list = list(tfds.as_numpy(step_ds))
    num_steps = len(steps_list)
    
    # Actions couvertes par les subtrajectories
    if ep_idx < len(mappingbis):
        num_covered = len([c for c in mappingbis[ep_idx] if c != -1])
        total_actions += num_steps
        total_covered_actions += num_covered
        
        coverage_pct = 100 * num_covered / num_steps if num_steps > 0 else 0
        status = "✅" if num_covered == num_steps else "⚠️"
        
        episode_details.append({
            'ep_idx': ep_idx,
            'total': num_steps,
            'covered': num_covered,
            'coverage_pct': coverage_pct
        })
        
        print(f"Episode {ep_idx}: {num_covered}/{num_steps} actions couvertes ({coverage_pct:.1f}%) {status}")
        
        # Détailler si certaines actions n'ont pas de cluster
        if num_covered < num_steps:
            missing_indices = [i for i, c in enumerate(mappingbis[ep_idx]) if c == 0]
            missing_str = str(missing_indices[:10]) if len(missing_indices) <= 10 else f"{missing_indices[:10]}... et {len(missing_indices)-10} autres"
            print(f"  ⚠️  {len(missing_indices)} actions MANQUANTES aux indices: {missing_str}")

print(f"\n{'='*80}")
print(f"RÉSULTAT GLOBAL: {total_covered_actions}/{total_actions} actions couvertes ({100*total_covered_actions/total_actions:.1f}%)")
print(f"{'='*80}")

# 2. Vérifier la distribution des cluster_ids
print("\n[2] DISTRIBUTION DES CLUSTER_IDS")
print("-" * 80)

all_cluster_ids = []
for ep_idx in range(len(mappingbis)):
    all_cluster_ids.extend([c for c in mappingbis[ep_idx] if c != 0])

if len(all_cluster_ids) > 0:
    all_cluster_ids_array = np.array(all_cluster_ids)
    unique_clusters = np.unique(all_cluster_ids_array)
    print(f"Nombre de clusters uniques: {len(unique_clusters)}")
    print(f"Plage des clusters: [{np.min(all_cluster_ids_array)}, {np.max(all_cluster_ids_array)}]")
    print(f"Clusters attendus: 0-29 (30 clusters)")
    
    if np.max(all_cluster_ids_array) >= 30:
        print(f"\n⚠️  ERROR: Cluster IDs dépassent 30!")
        print(f"   Clusters trouvés >= 30: {unique_clusters[unique_clusters >= 30].tolist()}")
    
    # Distribution
    counts = np.bincount(all_cluster_ids_array)
    print(f"\nDistribution des clusters:")
    for cid in range(min(30, len(counts))):
        count = counts[cid] if cid < len(counts) else 0
        pct = 100 * count / len(all_cluster_ids_array) if len(all_cluster_ids_array) > 0 else 0
        bar = "█" * max(1, count // max(1, np.max(counts)//30)) if count > 0 else ""
        print(f"  Cluster {cid:2d}: {count:4d} ({pct:5.1f}%) {bar}")
else:
    print("⚠️  ERREUR: Aucun cluster_id trouvé!")

# 3. Vérifier le dataset TFDS généré
print("\n[3] VÉRIFICATION DU DATASET TFDS GÉNÉRÉ")
print("-" * 80)

data_dir = "/home/ids/ext-5219/tokenizer/test/columbia_cairlab_pusht_real/1.0.0/"

try:
    builder_test = tfds.builder_from_directory(data_dir)
    ds_tfds = builder_test.as_dataset(split='train')
    
    for ep_idx, episode in enumerate(ds_tfds.take(3)):
        steps = episode['steps']
        cluster_ids_tfds = steps['observation']['cluster_id'].numpy()
        num_steps = len(cluster_ids_tfds)
        num_non_zero = np.sum(cluster_ids_tfds != 0)
        
        print(f"\nEpisode {ep_idx} dans TFDS:")
        print(f"  Nombre de steps: {num_steps}")
        print(f"  Nombre de cluster_ids non-zéro: {num_non_zero}")
        print(f"  Couverture: {100*num_non_zero/num_steps:.1f}%")
        print(f"  Min cluster: {np.min(cluster_ids_tfds)}, Max cluster: {np.max(cluster_ids_tfds)}")
        
        # Vérifier les valeurs par défaut
        zeros = np.sum(cluster_ids_tfds == -1)
        if zeros > 0:
            print(f"  ⚠️  {zeros} actions avec cluster_id = 0 (valeur par défaut)!")
            zero_indices = np.where(cluster_ids_tfds == -1)[0]
            zero_str = f"{zero_indices[:20].tolist()}" if len(zero_indices) > 20 else f"{zero_indices.tolist()}"
            print(f"     Indices: {zero_str}")

except Exception as e:
    print(f"❌ Erreur lors de la lecture du TFDS: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*80)
print("FIN DU DIAGNOSTIC")
print("="*80)

DIAGNOSTIC COMPLET DU DATASET
✅ mappingbis chargé depuis cluster_ids.pkl

[1] VÉRIFICATION DE LA COUVERTURE DES SEGMENTS
--------------------------------------------------------------------------------
Episode 0: 163/163 actions couvertes (100.0%) ✅
Episode 1: 188/188 actions couvertes (100.0%) ✅
Episode 2: 271/271 actions couvertes (100.0%) ✅
Episode 3: 158/158 actions couvertes (100.0%) ✅
Episode 4: 200/200 actions couvertes (100.0%) ✅
Episode 5: 237/237 actions couvertes (100.0%) ✅
Episode 6: 174/174 actions couvertes (100.0%) ✅
Episode 7: 177/177 actions couvertes (100.0%) ✅
Episode 8: 236/236 actions couvertes (100.0%) ✅
Episode 9: 125/125 actions couvertes (100.0%) ✅
Episode 10: 227/227 actions couvertes (100.0%) ✅
Episode 11: 195/195 actions couvertes (100.0%) ✅
Episode 12: 216/216 actions couvertes (100.0%) ✅
Episode 13: 210/210 actions couvertes (100.0%) ✅
Episode 14: 232/232 actions couvertes (100.0%) ✅
Episode 15: 262/262 actions couvertes (100.0%) ✅
Episode 16: 127/127 acti

Episode 121: 147/147 actions couvertes (100.0%) ✅

RÉSULTAT GLOBAL: 24924/24924 actions couvertes (100.0%)

[2] DISTRIBUTION DES CLUSTER_IDS
--------------------------------------------------------------------------------
Nombre de clusters uniques: 29
Plage des clusters: [1, 29]
Clusters attendus: 0-29 (30 clusters)

Distribution des clusters:
  Cluster  0:    0 (  0.0%) 
  Cluster  1:  867 (  3.5%) ██████
  Cluster  2:  877 (  3.5%) ██████
  Cluster  3:  394 (  1.6%) ███
  Cluster  4:  786 (  3.2%) ██████
  Cluster  5: 1829 (  7.3%) ██████████████
  Cluster  6:  216 (  0.9%) █
  Cluster  7: 3868 ( 15.5%) ██████████████████████████████
  Cluster  8:  465 (  1.9%) ███
  Cluster  9:  265 (  1.1%) ██
  Cluster 10:  371 (  1.5%) ██
  Cluster 11:  996 (  4.0%) ███████
  Cluster 12:  942 (  3.8%) ███████
  Cluster 13:  212 (  0.9%) █
  Cluster 14:  812 (  3.3%) ██████
  Cluster 15: 1187 (  4.8%) █████████
  Cluster 16:  492 (  2.0%) ███
  Cluster 17:  681 (  2.7%) █████
  Cluster 18:  355 (

2026-02-13 15:32:08.725462: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.



Episode 0 dans TFDS:
  Nombre de steps: 1
  Nombre de cluster_ids non-zéro: 0
  Couverture: 0.0%
  Min cluster: 0, Max cluster: 0
  ⚠️  1 actions avec cluster_id = 0 (valeur par défaut)!
     Indices: [0]

Episode 1 dans TFDS:
  Nombre de steps: 1
  Nombre de cluster_ids non-zéro: 0
  Couverture: 0.0%
  Min cluster: 0, Max cluster: 0
  ⚠️  1 actions avec cluster_id = 0 (valeur par défaut)!
     Indices: [0]

Episode 2 dans TFDS:
  Nombre de steps: 1
  Nombre de cluster_ids non-zéro: 0
  Couverture: 0.0%
  Min cluster: 0, Max cluster: 0
  ⚠️  1 actions avec cluster_id = 0 (valeur par défaut)!
     Indices: [0]

FIN DU DIAGNOSTIC
